In [1]:
import numpy as np
import pandas as pd
import itertools
from collections import defaultdict
from typing import Dict, Tuple, Any, List
from scipy import stats
from statsmodels.stats.power import TTestIndPower
import warnings
import os

try:
    from tqdm.auto import tqdm  
except Exception:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else range(0)


os.chdir('../../..')

from Scripts.Spectral_Analysis.Spectrum_Filter import filter_spectras
warnings.filterwarnings("ignore")

In [2]:
pwd

'/Data/EEG-Visual-Experiment'

In [5]:
# =========================
#  Memory-safe analysis cell
# =========================

from __future__ import annotations

import gc
import itertools
from collections import defaultdict
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from scipy import stats
from statsmodels.stats.power import TTestIndPower


# --------- группировки ---------

FEATURE_GROUPS = {
    "day_time": {
        "Day":      ["Day"],
        "Evening":  ["Evening"],
    },
    "stim_type": {
        "r": ["r"],
        "g": ["g"],
    },
    "stim_label": {
        "line":   [0, 1, 4, 6, 7, 11, 8],
        "figure": [2, 3, 5, 9, 10, 12],
    },
    "gender": {
        "m": ["m"],
        "f": ["f"],
    },
    "age": {
        "18-22": [18, 19, 20, 21, 22],
        "23-29": [23, 24, 25, 25, 26, 27, 28, 29],
        "30-35": [30, 31, 32, 33, 34, 35],
    },
    "handiness": {
        "l": ["l"],
        "r": ["r"],
    },
}

DEFAULT_BANDS = {
    "Delta": (1, 4),
    "Tetta": (4, 7),
    "Alpha": (7, 13),
    "Beta":  (13, 30),
}

_RESULT_FIELDS = [
    "power",
    "phase",
    "s_id",
    "t_id",
    "gender",
    "handiness",
    "age",
    "label",
    "img",
    "task_type",
    "day_time",
    "stim_type",
]


# --------- normalize ---------

def normalize_results(raw_results: List[Any], show_progress: bool = False) -> List[Dict[str, Any]]:
    """
    Приводит вывод filter_spectras(...) к list[dict] с гарантированными ключами.
    """
    norm: List[Dict[str, Any]] = []
    it = raw_results
    if show_progress:
        it = tqdm(raw_results, desc="Normalizing trials", leave=False)

    for item in it:
        if isinstance(item, dict):
            rec = item.copy()
        else:
            rec = {}
            for idx, key in enumerate(_RESULT_FIELDS):
                if idx < len(item):
                    rec[key] = item[idx]

        # новый формат: subject_id / trial_id -> s_id / t_id
        if "subject_id" in rec and "s_id" not in rec:
            rec["s_id"] = rec["subject_id"]
        if "trial_id" in rec and "t_id" not in rec:
            rec["t_id"] = rec["trial_id"]

        for key in _RESULT_FIELDS:
            rec.setdefault(key, None)

        norm.append(rec)

    return norm


def _is_normalized_trials(x: Any) -> bool:
    """
    Быстрый хак: если это list[dict] и у dict есть 'power' -> считаем нормализованным.
    """
    if not isinstance(x, list):
        return False
    if len(x) == 0:
        return True
    if not isinstance(x[0], dict):
        return False
    return ("power" in x[0])  # достаточно для наших целей


def _ensure_normalized(x: Any) -> List[Dict[str, Any]]:
    return x if _is_normalized_trials(x) else normalize_results(x, show_progress=False)


# --------- bands ---------

def make_band_cols_from_bands(
    bands: Dict[str, Tuple[float, float]],
    freqs: np.ndarray,
) -> Dict[str, np.ndarray]:
    band_cols: Dict[str, np.ndarray] = {}
    for band_name, (f_lo, f_hi) in bands.items():
        idx = np.where((freqs >= f_lo) & (freqs <= f_hi))[0]
        band_cols[band_name] = idx
    return band_cols


# --------- aggregation (LOW MEM) ---------

def aggregate_by_subject(
    results: List[Dict[str, Any]],
    band_cols: Dict[str, np.ndarray],
    n_channels: int,
    clear_power: bool = True,
) -> Dict[Tuple[int, str], Dict[str, Any]]:
    """
    По-триальная агрегация (каждый trial — независимое наблюдение), low-mem.

    ВАЖНО: мы НЕ создаём промежуточный vecs(C,F) на trial.
    Считаем среднее по бэнду (и времени, если есть T) напрямую из power.
    """
    trial_vecs: Dict[Tuple[int, str], Dict[str, Any]] = defaultdict(
        lambda: {"ids": [], "s_ids": [], "vals": []}
    )

    for rec in results:
        power = rec.get("power", None)
        if power is None:
            continue

        s_id = rec.get("s_id", rec.get("subject_id", None))
        t_id = rec.get("t_id", rec.get("trial_id", None))

        try:
            sid_int = int(s_id) if s_id is not None else None
        except Exception:
            sid_int = None
        try:
            tid_int = int(t_id) if t_id is not None else None
        except Exception:
            tid_int = None

        arr = np.asarray(power)  # без dtype, чтобы не копировать весь массив
        if arr.ndim not in (2, 3):
            if clear_power:
                rec["power"] = None
            continue

        max_ch = min(n_channels, arr.shape[0])
        n_freqs = arr.shape[1]

        for ch in range(max_ch):
            for band_name, freq_idx in band_cols.items():
                if freq_idx.size == 0:
                    continue
                if np.max(freq_idx) >= n_freqs:
                    continue

                if arr.ndim == 3:
                    # (len(freq_idx), T) -> scalar
                    band_val = float(np.nanmean(arr[ch, freq_idx, :]))
                else:
                    # (len(freq_idx),) -> scalar
                    band_val = float(np.nanmean(arr[ch, freq_idx]))

                key = (ch, band_name)
                trial_vecs[key]["ids"].append(tid_int)
                trial_vecs[key]["s_ids"].append(sid_int)
                trial_vecs[key]["vals"].append(band_val)

        # критично: убрать ссылку на большой массив, чтобы не держать RAM дальше
        if clear_power:
            rec["power"] = None

    for key, d in trial_vecs.items():
        d["vals"] = np.asarray(d["vals"], dtype=float)

    return trial_vecs


# --------- vectorization ---------

def build_two_condition_vectors(
    condA_subj: Dict[Tuple[int, str], Dict[str, Any]],
    condB_subj: Dict[Tuple[int, str], Dict[str, Any]],
    condA_name: str,
    condB_name: str,
    n_channels: int,
    band_names: List[str],
) -> Dict[Tuple[int, str], Dict[str, Any]]:
    subject_vectors: Dict[Tuple[int, str], Dict[str, Any]] = {}

    for ch in range(n_channels):
        for band_name in band_names:
            key = (ch, band_name)

            A = condA_subj.get(key, {"ids": [], "s_ids": [], "vals": np.array([], float)})
            B = condB_subj.get(key, {"ids": [], "s_ids": [], "vals": np.array([], float)})

            x = np.asarray(A.get("vals", np.array([], float)), dtype=float)
            y = np.asarray(B.get("vals", np.array([], float)), dtype=float)

            subject_vectors[key] = {
                condA_name: x,
                condB_name: y,
                f"{condA_name}_ids": list(A.get("ids", [])),
                f"{condB_name}_ids": list(B.get("ids", [])),
                f"{condA_name}_s_ids": list(A.get("s_ids", [])),
                f"{condB_name}_s_ids": list(B.get("s_ids", [])),
            }

    return subject_vectors


# --------- stats ---------

def hedges_g(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return np.nan

    vx, vy = np.var(x, ddof=1), np.var(y, ddof=1)

    if vx == 0 and vy == 0:
        return float(np.sign(np.nanmean(x) - np.nanmean(y)) * 0.0)

    sp2 = ((nx - 1) * vx + (ny - 1) * vy) / (nx + ny - 2)
    if sp2 <= 0:
        return np.nan

    d = (np.nanmean(x) - np.nanmean(y)) / np.sqrt(sp2)
    J = 1 - 3 / (4 * (nx + ny) - 9) if (nx + ny) > 2 else 1.0
    return float(J * d)


def build_band_table_two_conditions(
    subject_vectors: Dict[Tuple[int, str], Dict[str, Any]],
    condA_name: str,
    condB_name: str,
    alpha: float = 0.05,
) -> pd.DataFrame:
    try:
        import statsmodels.api as sm
        _HAS_SM = True
    except Exception:
        _HAS_SM = False

    power_calc = TTestIndPower()
    rows = []

    items = list(subject_vectors.items())
    for (ch, band_name), vecs in tqdm(
        items,
        total=len(items),
        desc=f"Stats per ch×band ({condA_name} vs {condB_name})",
        leave=True,
        miniters=1,
        mininterval=0.0,
        dynamic_ncols=True,
    ):
        x = np.asarray(vecs.get(condA_name, []), float)
        y = np.asarray(vecs.get(condB_name, []), float)
        nA, nB = len(x), len(y)
        if nA == 0 or nB == 0:
            continue

        sA = np.asarray(vecs.get(f"{condA_name}_s_ids", []))
        sB = np.asarray(vecs.get(f"{condB_name}_s_ids", []))

        # Welch по умолчанию
        t_stat, p_val = np.nan, np.nan
        try:
            t_stat, p_val = stats.ttest_ind(x, y, equal_var=False, nan_policy="omit")
        except Exception:
            pass

        g = hedges_g(x, y)

        used_cluster = False
        if _HAS_SM and (sA.size == nA) and (sB.size == nB):
            mA = pd.notna(sA)
            mB = pd.notna(sB)
            x2 = x[mA]; y2 = y[mB]
            sA2 = sA[mA]; sB2 = sB[mB]

            if (
                len(x2) >= 2 and len(y2) >= 2
                and len(np.unique(sA2)) >= 2 and len(np.unique(sB2)) >= 2
            ):
                y_all = np.concatenate([x2, y2])
                g_all = np.concatenate([np.zeros_like(x2), np.ones_like(y2)])
                sid_all = np.concatenate([sA2, sB2])

                try:
                    X = sm.add_constant(g_all.astype(float))
                    model = sm.OLS(y_all, X, missing="drop")
                    res = model.fit(cov_type="cluster", cov_kwds={"groups": sid_all})
                    t_stat = float(res.tvalues[1])
                    p_val = float(res.pvalues[1])
                    used_cluster = True

                    # power по числу кластеров
                    dfA = pd.DataFrame({"sid": sA2, "val": x2}).groupby("sid")["val"].mean()
                    dfB = pd.DataFrame({"sid": sB2, "val": y2}).groupby("sid")["val"].mean()
                    nA_eff = len(dfA)
                    nB_eff = len(dfB)
                    if nA_eff >= 2 and nB_eff >= 2:
                        g_eff = hedges_g(dfA.values, dfB.values)
                        try:
                            ratio = nB_eff / max(nA_eff, 1)
                            power = power_calc.solve_power(
                                effect_size=abs(g_eff),
                                nobs1=nA_eff,
                                ratio=ratio,
                                alpha=alpha,
                                alternative="two-sided",
                            )
                        except Exception:
                            power = np.nan
                    else:
                        power = np.nan

                except Exception:
                    used_cluster = False

        if not used_cluster:
            try:
                ratio = nB / max(nA, 1)
                power = power_calc.solve_power(
                    effect_size=abs(g),
                    nobs1=nA,
                    ratio=ratio,
                    alpha=alpha,
                    alternative="two-sided",
                )
            except Exception:
                power = np.nan

        rows.append(
            {
                "channel": ch,
                "band": band_name,
                f"n_{condA_name}": nA,
                f"n_{condB_name}": nB,
                f"mean_{condA_name}": float(np.nanmean(x)) if nA else np.nan,
                f"mean_{condB_name}": float(np.nanmean(y)) if nB else np.nan,
                f"delta_{condA_name}_minus_{condB_name}": float(np.nanmean(x) - np.nanmean(y))
                if (nA and nB) else np.nan,
                "t_stat": float(t_stat) if np.isfinite(t_stat) else np.nan,
                "p_value": float(p_val) if np.isfinite(p_val) else np.nan,
                "hedges_g": float(g) if np.isfinite(g) else np.nan,
                "power": float(power) if np.isfinite(power) else np.nan,
                "sig_alpha_0.05": bool((p_val <= 0.05) if np.isfinite(p_val) else False),
                "sig_alpha_0.01": bool((p_val <= 0.01) if np.isfinite(p_val) else False),
                "sig_and_power": bool((p_val <= 0.05) and (power >= 0.8))
                if (np.isfinite(p_val) and np.isfinite(power)) else False,
                "clustered": used_cluster,
            }
        )

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["p_value", "power"], ascending=[True, True]).reset_index(drop=True)
    return df


def run_stats_between_groups(
    data_A_raw,
    data_B_raw,
    group_A_name: str,
    group_B_name: str,
    band_cols: Dict[str, np.ndarray],
    n_channels: int,
    alpha: float = 0.05,
    clear_power: bool = True,
):
    # нормализуем только если нужно
    data_A = _ensure_normalized(data_A_raw)
    data_B = _ensure_normalized(data_B_raw)

    A_by = aggregate_by_subject(data_A, band_cols=band_cols, n_channels=n_channels, clear_power=clear_power)
    B_by = aggregate_by_subject(data_B, band_cols=band_cols, n_channels=n_channels, clear_power=clear_power)

    # облегчить жизнь GC (внутри процесса)
    del data_A, data_B
    gc.collect()

    band_names = list(band_cols.keys())
    subj_vecs = build_two_condition_vectors(
        condA_subj=A_by,
        condB_subj=B_by,
        condA_name=group_A_name,
        condB_name=group_B_name,
        n_channels=n_channels,
        band_names=band_names,
    )

    stats_table = build_band_table_two_conditions(
        subject_vectors=subj_vecs,
        condA_name=group_A_name,
        condB_name=group_B_name,
        alpha=alpha,
    )

    return {"summary_table": stats_table, "meta": {"group_A": group_A_name, "group_B": group_B_name}}


# --------- filtering ---------

def _collapse_allowed_values(allowed_values: List[Any]):
    return allowed_values[0] if len(allowed_values) == 1 else allowed_values


def _has_feature_in_trials(trials: List[Dict[str, Any]], feature_name: str) -> bool:
    for tr in trials:
        if isinstance(tr, dict) and (feature_name in tr) and (tr[feature_name] is not None):
            return True
    return False


def filter_group(
    npz_path: str,
    feature_name: str,
    allowed_values: List[Any],
    extra_kwargs: dict,
):
    """
    1) Если есть кэш _all_trials и в нём реально есть это поле -> режем в памяти.
    2) Иначе -> filter_spectras(...) и normalize_results(...).
    """
    if extra_kwargs is None:
        extra_kwargs = {}

    all_trials = extra_kwargs.get("_all_trials")
    if (all_trials is not None) and _has_feature_in_trials(all_trials, feature_name):
        allowed = set(allowed_values)
        out = []
        for tr in all_trials:
            val = tr.get(feature_name, None)
            if val is None:
                continue

            if isinstance(val, (list, tuple, np.ndarray)):
                if any(v in allowed for v in val):
                    out.append(tr)
            else:
                if val in allowed:
                    out.append(tr)
        return out

    kwargs = {"exec_spec_path": npz_path}

    meta_path = extra_kwargs.get("day_time_meta_path")
    if meta_path is not None:
        kwargs["day_time_meta_path"] = meta_path

    val = _collapse_allowed_values(allowed_values)

    if feature_name == "day_time":
        kwargs["day_time"] = val
    elif feature_name == "stim_type":
        kwargs["stim_type"] = val
    elif feature_name == "stim_label":
        kwargs["stim_label"] = val
    elif feature_name == "gender":
        kwargs["gender"] = val
    elif feature_name == "age":
        kwargs["age"] = val
    elif feature_name == "handiness":
        kwargs["handiness"] = val
    elif feature_name == "task_type":
        kwargs["task_type"] = val
    else:
        raise ValueError(f"Unknown feature: {feature_name}")

    raw = filter_spectras(**kwargs)   # <-- функция должна быть определена у тебя в ноутбуке
    return normalize_results(raw, show_progress=False)


# --------- session cfg ---------

def prepare_session_config_for_npz(
    npz_path: str,
    bands: Optional[Dict[str, Tuple[float, float]]] = None,
    extra_kwargs: Optional[dict] = None,
):
    """
    cache_all_trials=False:
      - _all_trials=None
      - форму power берём через np.load(..., mmap_mode="r")

    cache_all_trials=True:
      - грузим ВСЁ через filter_spectras
      - кладём нормализованные trials в extra_kwargs["_all_trials"]
    """
    if bands is None:
        bands = DEFAULT_BANDS.copy()
    if extra_kwargs is None:
        extra_kwargs = {}
    extra_kwargs = dict(extra_kwargs)

    cache_all = bool(extra_kwargs.get("cache_all_trials", False))

    example_power = None
    if cache_all:
        base_kwargs = {"exec_spec_path": npz_path}
        if "day_time_meta_path" in extra_kwargs:
            base_kwargs["day_time_meta_path"] = extra_kwargs["day_time_meta_path"]

        all_trials_raw = filter_spectras(**base_kwargs)
        all_trials_norm = normalize_results(all_trials_raw, show_progress=False)
        extra_kwargs["_all_trials"] = all_trials_norm

        for rec in all_trials_norm:
            if rec.get("power") is not None:
                example_power = rec["power"]
                break

        if example_power is None:
            raise RuntimeError("Не удалось найти ни одного trial с 'power' (после filter_spectras).")
    else:
        extra_kwargs["_all_trials"] = None
        with np.load(npz_path, mmap_mode="r") as npz:
            power_keys = [k for k in npz.files if k.startswith("power_")]
            if not power_keys:
                raise RuntimeError(f"В {npz_path} нет массивов 'power_*'")
            example_power = npz[power_keys[0]]

    example_power = np.asarray(example_power)
    if example_power.ndim == 3:
        n_channels, n_freqs, _ = example_power.shape
    elif example_power.ndim == 2:
        n_channels, n_freqs = example_power.shape
    else:
        raise ValueError(f"Неожиданная форма 'power': {getattr(example_power, 'shape', None)}")

    freqs = np.linspace(2, 40, n_freqs)
    band_cols = make_band_cols_from_bands(bands, freqs)

    return {
        "npz_path": npz_path,
        "n_channels": int(n_channels),
        "band_cols": band_cols,
        "bands": bands,
        "extra_kwargs": extra_kwargs,
    }


# --------- analysis (STREAMING / LOW MEM) ---------

def analyze_single_feature(session_cfg: dict, feature_name: str, alpha: float = 0.05):
    npz_path = session_cfg["npz_path"]
    band_cols = session_cfg["band_cols"]
    n_channels = session_cfg["n_channels"]
    extra_kwargs = session_cfg["extra_kwargs"]

    # если есть общий кэш trials -> НЕ трогаем power (иначе сломаешь кэш)
    clear_power = (extra_kwargs.get("_all_trials") is None)

    group_spec = FEATURE_GROUPS[feature_name]
    group_labels = list(group_spec.keys())

    results_for_feature = {}

    # стримингово: не храним все группы сразу
    for g1, g2 in itertools.combinations(group_labels, 2):
        data_A = filter_group(npz_path, feature_name, group_spec[g1], extra_kwargs)
        data_B = filter_group(npz_path, feature_name, group_spec[g2], extra_kwargs)

        results_for_feature[(g1, g2)] = run_stats_between_groups(
            data_A_raw=data_A,
            data_B_raw=data_B,
            group_A_name=g1,
            group_B_name=g2,
            band_cols=band_cols,
            n_channels=n_channels,
            alpha=alpha,
            clear_power=clear_power,
        )

        del data_A, data_B
        gc.collect()

    return results_for_feature


def analyze_feature_pair(session_cfg: dict, feature_A: str, feature_B: str, alpha: float = 0.05):
    npz_path = session_cfg["npz_path"]
    band_cols = session_cfg["band_cols"]
    n_channels = session_cfg["n_channels"]
    extra_kwargs = session_cfg["extra_kwargs"]

    clear_power = (extra_kwargs.get("_all_trials") is None)

    groups_A = FEATURE_GROUPS[feature_A]
    groups_B = FEATURE_GROUPS[feature_B]

    results_for_pair = {}

    # low-mem: держим 1 группу A и 1 группу B одновременно
    for label_A, allowed_A in groups_A.items():
        data_A = filter_group(npz_path, feature_A, allowed_A, extra_kwargs)

        for label_B, allowed_B in groups_B.items():
            data_B = filter_group(npz_path, feature_B, allowed_B, extra_kwargs)

            results_for_pair[(label_A, label_B)] = run_stats_between_groups(
                data_A_raw=data_A,
                data_B_raw=data_B,
                group_A_name=label_A,
                group_B_name=label_B,
                band_cols=band_cols,
                n_channels=n_channels,
                alpha=alpha,
                clear_power=clear_power,
            )

            del data_B
            gc.collect()

        del data_A
        gc.collect()

    return results_for_pair


def run_full_analysis_for_npz(
    npz_path: str,
    alpha: float = 0.05,
    feature_list_single=None,
    feature_pairs=None,
    extra_kwargs: Optional[dict] = None,
    bands: Optional[Dict[str, Tuple[float, float]]] = None,
):
    if extra_kwargs is None:
        extra_kwargs = {}

    session_cfg = prepare_session_config_for_npz(
        npz_path=npz_path,
        bands=bands,
        extra_kwargs=extra_kwargs,
    )

    if feature_list_single is None:
        feature_list_single = list(FEATURE_GROUPS.keys())

    if feature_pairs is None:
        feature_pairs = list(itertools.combinations(FEATURE_GROUPS.keys(), 2))

    final_results = {"single_feature": {}, "pair_feature": {}}

    for feat in tqdm(feature_list_single, desc="Single-feature analysis"):
        final_results["single_feature"][feat] = analyze_single_feature(session_cfg, feat, alpha)

    for featA, featB in tqdm(feature_pairs, desc="Pair-feature analysis"):
        final_results["pair_feature"][(featA, featB)] = analyze_feature_pair(session_cfg, featA, featB, alpha)

    return final_results


In [10]:
import os
import warnings
import itertools
from multiprocessing import Pool, cpu_count

from tqdm.auto import tqdm

# ---- глобали воркера (живут внутри каждого процесса) ----
_WORKER_SESSION_CFG = None
_WORKER_ALPHA = None


def _force_blas_threads(n: int = 1) -> None:
    n = int(n)
    for var in (
        "OMP_NUM_THREADS",
        "MKL_NUM_THREADS",
        "OPENBLAS_NUM_THREADS",
        "NUMEXPR_NUM_THREADS",
        "VECLIB_MAXIMUM_THREADS",
    ):
        os.environ[var] = str(n)


def _init_pool_worker(session_cfg: dict, alpha: float, blas_threads: int = 1) -> None:
    global _WORKER_SESSION_CFG, _WORKER_ALPHA
    _force_blas_threads(blas_threads)
    _WORKER_SESSION_CFG = session_cfg
    _WORKER_ALPHA = alpha


def _worker_single_feature(feature_name: str):
    res = analyze_single_feature(
        session_cfg=_WORKER_SESSION_CFG,
        feature_name=feature_name,
        alpha=_WORKER_ALPHA,
    )
    return feature_name, res


def _worker_feature_pair(pair):
    feature_A, feature_B = pair
    res = analyze_feature_pair(
        session_cfg=_WORKER_SESSION_CFG,
        feature_A=feature_A,
        feature_B=feature_B,
        alpha=_WORKER_ALPHA,
    )
    return (feature_A, feature_B), res


def run_full_analysis_for_npz_parallel(
    npz_path: str,
    alpha: float = 0.05,
    feature_list_single=None,
    feature_pairs=None,
    extra_kwargs: dict = None,
    bands: dict = None,
    n_jobs: int = 8,
    maxtasksperchild: int = 10,
    blas_threads: int = 1,
    chunksize: int | None = None,
    progress: bool = True,
):
    """
    Memory-safe parallel анализ одного npz + "живой" прогресс-бар:
      - session_cfg/alpha кладём в init воркера (1 раз), не в каждую задачу
      - maxtasksperchild перезапускает воркеры и реально освобождает RSS
      - ограничиваем BLAS threads
      - отдельный Pool для single и для pair
      - tqdm обновляется часто: по умолчанию chunksize=1
    """
    if extra_kwargs is None:
        extra_kwargs = {}
    extra_kwargs = dict(extra_kwargs)

    _force_blas_threads(blas_threads)

    session_cfg = prepare_session_config_for_npz(
        npz_path=npz_path,
        bands=bands,
        extra_kwargs=extra_kwargs,
    )

    has_day_time_meta = "day_time_meta_path" in extra_kwargs

    if feature_list_single is None:
        feature_list_single = list(FEATURE_GROUPS.keys())

    if feature_pairs is None:
        feature_pairs = list(itertools.combinations(FEATURE_GROUPS.keys(), 2))

    if not has_day_time_meta:
        feature_list_single = [f for f in feature_list_single if f != "day_time"]
        feature_pairs = [(a, b) for (a, b) in feature_pairs if a != "day_time" and b != "day_time"]

    n_jobs = max(1, min(int(n_jobs), cpu_count()))
    if n_jobs >= 16:
        warnings.warn(
            f"n_jobs={n_jobs} — это часто приводит к OOM на больших массивах. "
            f"Обычно стабильно 4..8 + maxtasksperchild."
        )

    final_results = {"single_feature": {}, "pair_feature": {}}

    # ---- SEQ fallback ----
    if n_jobs == 1:
        it1 = feature_list_single
        it2 = feature_pairs
        if progress:
            it1 = tqdm(it1, desc="Single-feature (seq)", unit="feat", dynamic_ncols=True)
            it2 = tqdm(it2, desc="Pair-feature (seq)", unit="pair", dynamic_ncols=True)

        for feat in it1:
            final_results["single_feature"][feat] = analyze_single_feature(session_cfg, feat, alpha)

        for a, b in it2:
            final_results["pair_feature"][(a, b)] = analyze_feature_pair(session_cfg, a, b, alpha)

        return final_results

    # ---- chunksize: чтобы прогресс шёл плавно ----
    if chunksize is None:
        # В Jupyter лучше 1: максимальная "живость" прогресса.
        chunksize_single = 1
        chunksize_pairs = 1
    else:
        chunksize_single = chunksize_pairs = int(chunksize)

    # ---- SINGLE: отдельный pool ----
    with Pool(
        processes=n_jobs,
        initializer=_init_pool_worker,
        initargs=(session_cfg, alpha, blas_threads),
        maxtasksperchild=maxtasksperchild,
    ) as pool:
        iterator = pool.imap_unordered(_worker_single_feature, feature_list_single, chunksize=chunksize_single)

        if progress:
            with tqdm(
                total=len(feature_list_single),
                desc="Single-feature (parallel)",
                unit="feat",
                dynamic_ncols=True,
                mininterval=0.1,
            ) as pbar:
                for feat, res in iterator:
                    final_results["single_feature"][feat] = res
                    pbar.set_postfix_str(f"done={feat}")
                    pbar.update(1)
        else:
            for feat, res in iterator:
                final_results["single_feature"][feat] = res

    # ---- PAIRS: новый pool ----
    with Pool(
        processes=n_jobs,
        initializer=_init_pool_worker,
        initargs=(session_cfg, alpha, blas_threads),
        maxtasksperchild=maxtasksperchild,
    ) as pool:
        iterator = pool.imap_unordered(_worker_feature_pair, feature_pairs, chunksize=chunksize_pairs)

        if progress:
            with tqdm(
                total=len(feature_pairs),
                desc="Pair-feature (parallel)",
                unit="pair",
                dynamic_ncols=True,
                mininterval=0.1,
            ) as pbar:
                for (a, b), res in iterator:
                    final_results["pair_feature"][(a, b)] = res
                    pbar.set_postfix_str(f"done={a}×{b}")
                    pbar.update(1)
        else:
            for (a, b), res in iterator:
                final_results["pair_feature"][(a, b)] = res

    return final_results


In [8]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def show_interesting_results(
    results,
    alpha=0.05,
    power_thresh=0.80,
    top=20,
    save_dir="./Supplementary",         # <-- по умолчанию туда
    sort_by=("p_value", "power"),       # p_value↑, power↓
    save_all_tables=True,               # <-- сохранять ВСЕ таблицы
    save_interesting_tables=True,       # <-- и отдельно interesting
):
    """
    Ищет и выводит «интересные» результаты по всем одиночным фичам и парам фичей:
      условие: (p_value <= alpha) & (power >= power_thresh).

    Дополнительно сохраняет таблицы в save_dir:
      - ALL__...csv (полная таблица)
      - INTERESTING__...csv (таблица после фильтра)

    Returns
    -------
    summary_df : pd.DataFrame
        объединенная таблица по interesting
    saved_files : List[str]
        какие файлы реально сохранились
    """
    rows = []
    saved_files = []

    save_path = Path(save_dir) if save_dir else None
    if save_path:
        save_path.mkdir(parents=True, exist_ok=True)

    def _safe_slug(s: str) -> str:
        s = str(s)
        s = s.strip()
        # заменяем пробелы и опасные символы на _
        s = re.sub(r"[^\w\.-]+", "_", s, flags=re.UNICODE)
        s = re.sub(r"_+", "_", s)
        return s[:180] if len(s) > 180 else s

    def _build_ascending(cols):
        asc = []
        for c in cols:
            if c == "power":
                asc.append(False)  # power по убыванию
            else:
                asc.append(True)   # остальные по возрастанию
        return asc

    def _save_df(df: pd.DataFrame, filename: str):
        if not save_path:
            return
        try:
            fn = _safe_slug(filename) + ".csv"
            full = save_path / fn
            df.to_csv(full, index=False)
            saved_files.append(str(full))
        except Exception as e:
            print(f"[WARN] Не удалось сохранить {filename}: {e}")

    def _append_rows(df, metaA, metaB, scope,
                     feature=None, feature_A=None, feature_B=None,
                     group_A=None, group_B=None):
        if df is None or df.empty:
            return

        def _col(name):
            return df[name] if (isinstance(df, pd.DataFrame) and (name in df.columns)) else np.nan

        base = pd.DataFrame({
            "scope": scope,
            "feature": feature,
            "feature_A": feature_A,
            "feature_B": feature_B,
            "group_A": group_A if group_A is not None else metaA,
            "group_B": group_B if group_B is not None else metaB,
            "channel": _col("channel"),
            "band":    _col("band"),
            "n_A":     _col(f"n_{metaA}"),
            "n_B":     _col(f"n_{metaB}"),
            "mean_A":  _col(f"mean_{metaA}"),
            "mean_B":  _col(f"mean_{metaB}"),
            "delta_A_minus_B": _col(f"delta_{metaA}_minus_{metaB}"),
            "t_stat":  _col("t_stat"),
            "p_value": _col("p_value"),
            "hedges_g":_col("hedges_g"),
            "power":   _col("power"),
            "sig_alpha_0.05": _col("sig_alpha_0.05"),
            "sig_alpha_0.01": _col("sig_alpha_0.01"),
            "sig_and_power":  _col("sig_and_power"),
            "clustered":      _col("clustered"),
        })
        rows.append(base)

    def _sort_df(df: pd.DataFrame) -> pd.DataFrame:
        if df is None or df.empty:
            return df
        existing_sort_cols = [c for c in sort_by if c in df.columns]
        if not existing_sort_cols:
            return df
        asc = _build_ascending(existing_sort_cols)
        return df.sort_values(existing_sort_cols, ascending=asc).copy()

    # -------- 1) одиночные фичи --------
    for feat, comp in results.get("single_feature", {}).items():
        for (gA, gB), entry in comp.items():
            df = entry.get("summary_table")
            if df is None or df.empty:
                continue

            # сохраняем полную таблицу
            if save_all_tables:
                _save_df(
                    _sort_df(df),
                    filename=f"ALL__single__{feat}__{gA}_vs_{gB}__alpha{alpha}_pwr{power_thresh}"
                )

            if ("p_value" not in df.columns) or ("power" not in df.columns):
                continue

            interesting = df[(df["p_value"] <= alpha) & (df["power"] >= power_thresh)].copy()
            if interesting.empty:
                continue

            interesting = _sort_df(interesting)

            print(f"\n{feat.upper()}: significant (p<= {alpha:.3g}, power>= {power_thresh:.2f}) — {gA} vs {gB}")
            display(interesting.head(top))

            if save_interesting_tables:
                _save_df(
                    interesting,
                    filename=f"INTERESTING__single__{feat}__{gA}_vs_{gB}__alpha{alpha}_pwr{power_thresh}"
                )

            metaA = entry.get("meta", {}).get("group_A", gA)
            metaB = entry.get("meta", {}).get("group_B", gB)
            _append_rows(interesting, metaA, metaB, scope="single", feature=feat, group_A=gA, group_B=gB)

    # -------- 2) пары фичей --------
    for (featA, featB), comp in results.get("pair_feature", {}).items():
        for (label_A, label_B), entry in comp.items():
            df = entry.get("summary_table")
            if df is None or df.empty:
                continue

            # сохраняем полную таблицу
            if save_all_tables:
                _save_df(
                    _sort_df(df),
                    filename=f"ALL__pair__{featA}x{featB}__{label_A}_vs_{label_B}__alpha{alpha}_pwr{power_thresh}"
                )

            if ("p_value" not in df.columns) or ("power" not in df.columns):
                continue

            interesting = df[(df["p_value"] <= alpha) & (df["power"] >= power_thresh)].copy()
            if interesting.empty:
                continue

            interesting = _sort_df(interesting)

            print(
                f"\n{featA.upper()} × {featB.upper()}: significant (p<= {alpha:.3g}, power>= {power_thresh:.2f}) "
                f"— {featA}={label_A} vs {featB}={label_B}"
            )
            display(interesting.head(top))

            if save_interesting_tables:
                _save_df(
                    interesting,
                    filename=f"INTERESTING__pair__{featA}x{featB}__{label_A}_vs_{label_B}__alpha{alpha}_pwr{power_thresh}"
                )

            metaA = entry.get("meta", {}).get("group_A", label_A)
            metaB = entry.get("meta", {}).get("group_B", label_B)
            _append_rows(
                interesting, metaA, metaB, scope="pair",
                feature_A=featA, feature_B=featB, group_A=label_A, group_B=label_B
            )

    # -------- итоговая сводка --------
    if rows:
        out = pd.concat(rows, ignore_index=True)

        for c in [
            "channel", "n_A", "n_B", "mean_A", "mean_B", "delta_A_minus_B",
            "t_stat", "p_value", "hedges_g", "power"
        ]:
            if c in out.columns:
                out[c] = pd.to_numeric(out[c], errors="coerce")

        out_sorted = _sort_df(out)

        print("\nИтоговая сводка (interesting) — первые строки")
        display(out_sorted.head(max(10, top)))

        return out_sorted, saved_files

    print("Интересных результатов не найдено при заданных порогах.")
    empty = pd.DataFrame(columns=[
        "scope", "feature", "feature_A", "feature_B", "group_A", "group_B",
        "channel", "band", "n_A", "n_B", "mean_A", "mean_B", "delta_A_minus_B",
        "t_stat", "p_value", "hedges_g", "power", "sig_alpha_0.05", "sig_alpha_0.01",
        "sig_and_power", "clustered"
    ])
    return empty, saved_files


In [11]:
# Путь к npz
npz_path = "./Generated/Spectrums/exec_and_rest_morlets.npz"

bands = {
    "Delta": (1, 4),
    "Theta": (4, 7),   # имя ключа любое, но чтобы не путаться лучше Theta
    "Alpha": (7, 13),
    "Beta":  (13, 30),
}

extra_kwargs = {
    "day_time_meta_path": "./Supplementary/Experiment_Metadata.xlsx",
    "cache_all_trials": False,   # <-- критично для памяти при multiprocessing
}

N_JOBS = 3

results = run_full_analysis_for_npz_parallel(
    npz_path=npz_path,
    alpha=0.05,
    bands=bands,
    extra_kwargs=extra_kwargs,
    n_jobs=N_JOBS,
    maxtasksperchild=5,
    blas_threads=1,
)



Single-feature (parallel):   0%|          | 0/6 [00:00<?, ?feat/s]

KeyboardInterrupt: 

In [12]:
import time

# 1) замер одной выборки (один filter_spectras внутри filter_group)
t0 = time.perf_counter()
_ = filter_group(npz_path, "gender", ["m"], extra_kwargs)
print("filter_group gender=m:", time.perf_counter() - t0, "sec")

# 2) замер одной тяжёлой фичи
session_cfg = prepare_session_config_for_npz(npz_path, bands=bands, extra_kwargs=extra_kwargs)

t0 = time.perf_counter()
_ = analyze_single_feature(session_cfg, "age", alpha=0.05)
print("analyze_single_feature age:", time.perf_counter() - t0, "sec")

# 3) замер одной тяжёлой пары
t0 = time.perf_counter()
_ = analyze_feature_pair(session_cfg, "age", "handiness", alpha=0.05)
print("analyze_feature_pair age×handiness:", time.perf_counter() - t0, "sec")


filter_group gender=m: 613.69102441 sec


KeyboardInterrupt: 